In [1]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
import os

In [ ]:
log_dir = "sb3_logs/"
model_dir = "sb3_models/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)


train_env = gym.make("LunarLander-v3")
eval_env = gym.make("LunarLander-v3")

In [ ]:
# Cell 1
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
import os

# Cell 2
log_dir = "sb3_logs/"     # ✅ Fixed: was 'og_dir'
model_dir = "sb3_models/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Try LunarLander, fallback to CartPole
try:
    train_env = gym.make("LunarLander-v3")
    eval_env = gym.make("LunarLander-v3")
    print("✅ LunarLander loaded successfully!")
except Exception as e:
    print(f"❌ LunarLander failed: {e}")
    print("🔄 Switching to CartPole...")
    train_env = gym.make("CartPole-v1")
    eval_env = gym.make("CartPole-v1")
    print("✅ CartPole loaded successfully!")

print(f"Training environment: {train_env.spec.id}")

# Cell 3 - Continue with your DQN training
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=model_dir,
    log_path=log_dir,
    eval_freq=10000,
    deterministic=True,
    render=False
)

model = DQN("MlpPolicy", train_env, verbose=1, tensorboard_log=log_dir)
model.learn(total_timesteps=100000, callback=eval_callback)

In [11]:
# Cell 1: Imports
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.callbacks import EvalCallback
import os

# Cell 2: Environment setup for LunarLander only
log_dir = "sb3_logs/"
model_dir = "sb3_models/"
os.makedirs(log_dir, exist_ok=True)
os.makedirs(model_dir, exist_ok=True)

# Create LunarLander environments
train_env = gym.make("LunarLander-v3")
eval_env = gym.make("LunarLander-v3")

print("✅ LunarLander environments created successfully!")
print(f"Observation space: {train_env.observation_space}")
print(f"Action space: {train_env.action_space}")

# Cell 3: Training setup and execution
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=model_dir,
    log_path=log_dir,
    eval_freq=10000,
    deterministic=True,
    render=False
)

# model = DQN("MlpPolicy", train_env, verbose=1, tensorboard_log=log_dir)
model = DQN(
    "MlpPolicy",
    train_env,
    learning_rate=1e-4,          # A slightly smaller learning rate
    buffer_size=50000,           # A smaller replay buffer might help it focus on more recent, relevant experience
    learning_starts=1000,
    batch_size=32,
    gamma=0.99,
    train_freq=(4, "step"),
    gradient_steps=1,
    target_update_interval=1000, # Update target network less frequently for more stability
    exploration_fraction=0.2,    # Explore for 20% of total timesteps
    exploration_final_eps=0.05,
    verbose=1
)

print("🚀 Starting LunarLander training...")
model.learn(total_timesteps=500000, callback=eval_callback)
print("✅ Training completed!")

# Cell 4: Test the trained model
print("🧪 Testing the trained model...")
obs, info = eval_env.reset()
total_reward = 0

for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    total_reward += reward
    
    if terminated or truncated:
        print(f"Episode completed with total reward: {total_reward:.2f}")
        obs, info = eval_env.reset()
        total_reward = 0
        break

eval_env.close()
print("✅ Testing completed!")

✅ LunarLander environments created successfully!
Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Action space: Discrete(4)
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🚀 Starting LunarLander training...
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 90.5     |
|    ep_rew_mean      | -219     |
|    exploration_rate | 0.997    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 2224     |
|    time_elapsed     | 0        |
|    total_timesteps  | 362      |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 87.2     |
|    ep_rew_mean      | -211     |
|    exploration_rate | 0.993    |
| time/             



---

### DQN Hyperparameter Deep Dive

#### **Core Learning Parameters**

1.  **`learning_rate=1e-4` (or `0.0001`)**
    *   **What it is:** The size of the step the algorithm takes to update the neural network's weights during training.
    *   **Analogy:** Imagine you're blindfolded on a bumpy hillside and trying to walk to the lowest point (the "minimum loss"). The `learning_rate` is your step size.
        *   **Too large:** You might take giant steps and completely leap over the lowest point, bouncing around erratically and never settling.
        *   **Too small:** You'll take tiny, shuffling steps. You'll eventually get to the bottom, but it might take an extremely long time.
    *   **Our Change:** We reduced it from the default (typically `1e-3`) to `1e-4`. We are telling the agent: "Be more careful and conservative with your updates. We think you're on a complex part of the 'hillside', so take smaller, more precise steps."

2.  **`gamma=0.99` (Discount Factor)**
    *   **What it is:** The "patience" or "farsightedness" of the agent. It determines how much current rewards are valued compared to future rewards. The value is always between 0 and 1.
    *   **Formula:** The total reward (Return) is calculated as `R_t = r_t + γ*r_{t+1} + γ²*r_{t+2} + ...`
    *   **Analogy:**
        *   `gamma = 0`: "I only care about the money I make *right now*." (A pure day trader).
        *   `gamma = 0.99`: "A dollar today is worth slightly more than a dollar tomorrow." The agent is willing to make sacrifices now for a bigger payoff later.
    *   **Our Choice:** `0.99` is a very standard value for problems with long time horizons. It encourages the agent to value the large +100 reward for landing, even if it's hundreds of steps away.

#### **Replay Buffer Parameters**

3.  **`buffer_size=50000`**
    *   **What it is:** The maximum number of past experiences `(state, action, reward, next_state)` to store in the Experience Replay memory.
    *   **Analogy:** It's the size of the deck of "flashcards" the agent studies from.
        *   **Too large:** The agent might waste time learning from very old, irrelevant experiences from when its strategy was terrible.
        *   **Too small:** The agent might not have enough diverse experiences to learn from and could start "overfitting" to its most recent actions.
    *   **Our Change:** We reduced it from the default (often `1,000,000`). Our hypothesis is that we want the agent to focus on more recent, higher-quality experiences now that it has learned the basics of not crashing.

4.  **`learning_starts=1000`**
    *   **What it is:** The number of steps the agent will take to just collect experiences *before* it ever starts training its neural network.
    *   **Analogy:** This is the "cram session" before the first exam. The agent fills its replay buffer with some initial random data so that the first training batch is diverse and not just a single, correlated sequence of steps.
    *   **Our Choice:** `1000` is a reasonable number to get a decent variety of initial states without waiting too long.

5.  **`batch_size=32`**
    *   **What it is:** The number of experiences randomly sampled from the replay buffer for each single training update.
    *   **Analogy:** If the replay buffer is a 50,000-card deck, the `batch_size` is the number of cards you pull out to study in one go.
    *   **Our Choice:** `32` is a very standard, well-tested batch size. It's a good balance between getting a representative sample of experiences and computational efficiency.

#### **Training Frequency Parameters**

6.  **`train_freq=(4, "step")`**
    *   **What it is:** How often to perform a training update.
    *   **Interpretation:** This says: "Perform one training update (using a batch of 32 experiences) every **4 steps** the agent takes in the environment."
    *   **Analogy:** This is the ratio of "acting" to "thinking." A value of `(1, "step")` means the agent thinks after every single action. A value of `(1, "episode")` means it only thinks after an entire episode is over.
    *   **Our Choice:** `(4, "step")` is a classic setting from the original DeepMind papers. It lets the agent collect a small amount of new data before pausing to learn.

7.  **`gradient_steps=1`**
    *   **What it is:** The number of training updates to perform when `train_freq` is triggered.
    *   **Interpretation:** When the 4-step trigger is hit, perform exactly **1** gradient update. If you set this to `100`, it would perform 100 updates (on 100 different mini-batches) every 4 steps. That would make the agent "think" a lot more than it "acts," which can sometimes be useful but is computationally expensive.
    *   **Our Choice:** `1` is the standard setting.

8.  **`target_update_interval=1000`**
    *   **What it is:** The frequency (in number of steps) at which we copy the weights from our main Q-network to the frozen "target" Q-network.
    *   **Analogy:** This controls how often the "answer key" (the target network) is updated with new information.
        *   **Too frequent:** The target changes too fast, and the main network is "chasing a moving target," leading to instability.
        *   **Too infrequent:** The target becomes stale, and the agent learns slowly from outdated information.
    *   **Our Choice:** We increased this from a smaller default. We are telling the agent: "Let's use a very stable, fixed target for a while to ensure your learning is consistent. We'll update it less often."

#### **Exploration/Exploitation Parameters**

9.  **`exploration_fraction=0.2`**
    *   **What it is:** The fraction of the *total training timesteps* over which the exploration rate (epsilon) will decrease.
    *   **Interpretation:** If `total_timesteps` is 200,000, the exploration rate will decay over the first `0.2 * 200,000 = 40,000` steps. For the first 40,000 steps, the agent will become progressively less random. After 40,000 steps, it will stick to its final, low exploration rate.
    *   **Our Change:** We increased this from a smaller default (e.g., 0.1). We are giving the agent more time to be "playful" and try random things before it has to settle down and exploit its knowledge. This might help it accidentally discover the high reward from landing.

10. **`exploration_final_eps=0.05`**
    *   **What it is:** The minimum value that the exploration rate (epsilon) will decay to.
    *   **Interpretation:** After the exploration fraction is over, the agent will still take a completely random action **5%** of the time. This ensures that the agent never becomes 100% deterministic and can always be surprised, potentially breaking out of a bad strategy even late in training.
    *   **Our Choice:** `0.05` is a standard value to ensure a small amount of lifelong exploration.

---
